In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().absolute().parent))

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
)
import matplotlib.pyplot as plt

import seaborn as sns

import pandas as pd
from pathlib import Path


from src.data_models.caravanify import Caravanify, CaravanifyConfig

from src.data_models.datamodule import HydroDataModule

from sklearn.pipeline import Pipeline

from src.preprocessing.grouped import GroupedTransformer

# from src.preprocessing.log_scale import LogTransformer
from src.preprocessing.standard_scale import StandardScaleTransformer

from src.model_evaluation.evaluators import TSForecastEvaluator
from src.model_evaluation.visualization import plot_rolling_forecast
from src.models.tide import LitTiDE, TiDEConfig

---

In [ ]:
config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CA",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)


caravan = Caravanify(config)
# ids_for_training = caravan.get_all_gauge_ids()[:10]
ids_for_training = ["CA_17110", "CA_17329"]

path_to_hii = "/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv"

ids_for_training, _ = caravan.filter_gauge_ids_by_human_influence(
    ids_for_training, ["Low", "Medium"]
)

caravan.load_stations(ids_for_training)
static_data = caravan.get_static_attributes()
ts_data = caravan.get_time_series()

In [ ]:
ts_columns = [
    "potential_evaporation_sum_FAO_PENMAN_MONTEITH",
    "streamflow",
    "temperature_2m_mean",
    "total_precipitation_sum",
]

ts_columns_to_keep = ts_columns + ["gauge_id", "date"]
ts_data = ts_data[ts_columns_to_keep]

In [ ]:
static_columns = [
    "p_mean",
    "area",
    "ele_mt_sav",
    "high_prec_dur",
    "frac_snow",
    "high_prec_freq",
    "slp_dg_sav",
    "cly_pc_sav",
    "aridity_ERA5_LAND",
    "aridity_FAO_PM",
]

static_columns_to_keep = static_columns + ["gauge_id"]

static_data = static_data[static_columns_to_keep]

In [ ]:
forcing_features = [
    col for col in ts_columns if col not in ["gauge_id", "date", "streamflow"]
]

target_feature = "streamflow"

---

In [ ]:
feature_pipeline = Pipeline(
    [
        # ("log", LogTransformer(columns=dynamic_feature_cols)),
        ("scaler", StandardScaleTransformer(columns=forcing_features)),
    ]
)

# Target pipeline: grouped by basin with log + scale
target_pipeline = GroupedTransformer(
    Pipeline(
        [
            # ("log", LogTransformer(columns=target_cols)),
            ("scaler", StandardScaleTransformer(columns=[target_feature])),
        ]
    ),
    columns=[target_feature],
    group_identifier="gauge_id",
    n_jobs=-1,
)

# Static feature pipeline: just scaling
static_pipeline = Pipeline(
    [("scaler", StandardScaleTransformer(columns=static_columns))]
)

# Define preprocessing configurations
preprocessing_configs = {
    "features": {"pipeline": feature_pipeline, "columns": ts_columns},
    "target": {"pipeline": target_pipeline, "columns": [target_feature]},
    "static_features": {"pipeline": static_pipeline, "columns": static_columns},
}

---

In [ ]:
output_length = 10
input_length = 40

data_module = HydroDataModule(
    time_series_df=ts_data,
    static_df=static_data,
    group_identifier="gauge_id",
    preprocessing_config=preprocessing_configs,
    batch_size=256,
    input_length=input_length,
    output_length=output_length,
    num_workers=4,
    features=ts_columns,
    static_features=static_columns,
    target="streamflow",
    train_prop=0.5,
    val_prop=0.25,
    test_prop=0.25,
    domain_id="CA",
    use_proportional_split=True,
    max_imputation_gap_size=2,
    min_train_years=5,
)

data_module.prepare_data()
data_module.setup(stage="fit")

---

In [ ]:
config = TiDEConfig(
    input_len=input_length,
    output_len=output_length,
    input_size=len(ts_columns) - 1,
    future_input_size=len(ts_columns) - 1,
    static_size=len(static_columns),
    num_encoder_layers=2,
    num_decoder_layers=2,
    decoder_output_size=16,
    hidden_size=16,
    temporal_decoder_hidden_size=16,
    past_feature_projection_size=4,
    future_forcing_projection_size=4,
    use_layer_norm=True,
    dropout=0.1,
    learning_rate=1e-3,
)

# Instantiate the Lightning module.
model = LitTiDE(config)

In [ ]:
model

In [ ]:
trainer = pl.Trainer(
    max_epochs=1,
    accelerator="cpu",
    devices=1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, mode="min"),
        LearningRateMonitor(logging_interval="epoch"),
    ],
)

# Train the model
trainer.fit(model, data_module)

In [ ]:
models_and_datamodules = {
    "TiDE": (model, data_module),
}

evaluator = TSForecastEvaluator(
    horizons=list(range(1, 11)),
    models_and_datamodules=models_and_datamodules,
    trainer_kwargs={"accelerator": "cpu", "devices": 1},
)

In [ ]:
results = evaluator.test_models()

In [ ]:
overall_summary = evaluator.summarize_metrics(results["TiDE"]["metrics"])


def plot_metric_summary(
    summary_df: pd.DataFrame, metric: str, per_basin: bool = False, figsize=(10, 6)
):
    plt.figure(figsize=figsize)

    if per_basin:
        df_plot = summary_df[metric].unstack(level=0)

        # Sort basins based on first horizon values
        first_horizon_values = df_plot.iloc[0]
        sorted_basins = first_horizon_values.sort_values(ascending=False).index
        df_plot = df_plot[sorted_basins]

        sns.barplot(
            data=df_plot.melt(ignore_index=False).reset_index(),
            x="horizon",
            y="value",
            hue="basin_id",
            palette="Blues",
        )
        plt.title(f"{metric} by Basin and Horizon")
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Basin ID")

    else:
        ax = sns.barplot(x=summary_df.index, y=summary_df[metric], color="steelblue")
        plt.title(f"Overall {metric} by Horizon")

        for i, v in enumerate(summary_df[metric]):
            ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")

    plt.xlabel("Forecast Horizon")
    plt.ylabel(metric)
    plt.tight_layout()
    sns.despine()
    plt.show()


# Plot overall NSE
plot_metric_summary(
    overall_summary,
    metric="NSE",
    per_basin=False,
    figsize=(10, 6),
)

In [ ]:
plot_rolling_forecast(
    results["TiDE"]["df"],
    group_identifier="CA_17110",
    horizon=1,
)

In [ ]:
def plot_streamflow(gauge_id, ts_data):
    plt.figure(figsize=(10, 5))
    plt.plot(ts_data[ts_data["gauge_id"] == gauge_id]["date"],
             ts_data[ts_data["gauge_id"] == gauge_id]["streamflow"])
    plt.title(f"Streamflow for {gauge_id}")
    plt.xlabel("Date")
    plt.ylabel("Streamflow (m^3/s)")
    plt.show()

In [ ]:
whole_df = data_module.processed_time_series
train = data_module.train_dataset.df_sorted
val = data_module.val_dataset.df_sorted
test = data_module.test_dataset.df_sorted

In [ ]:
plot_streamflow("CA_17329", train)
plot_streamflow("CA_17329", val)
plot_streamflow("CA_17329", test)
plot_streamflow("CA_17329", whole_df)

In [ ]:
whole_df = whole_df[whole_df["gauge_id"] == "CA_17110"]

In [ ]:
whole_df = whole_df.sort_values("date").reset_index(drop=True)


# Identify valid data points (not NaN for target)
valid_mask = ~whole_df["streamflow"].isna()

# Get indices of valid data points
valid_indices = whole_df.index[valid_mask].tolist()
n_valid = len(valid_indices)

n_valid

In [ ]:
5235 / 365